In [ ]:
%load_ext autoreload
%autoreload 2

import json
import numpy as np
import polars as pl
import pandas as pd
import pickle as pkl
from tqdm import tqdm

from plotnine import *
import matplotlib.pyplot as plt

## Samples

In [ ]:
pkl_file = '/s/project/deeprvat/deeprvat_input/data_500k/3rd_degree_unrelated_caucasian_sample_500k_ukb673180ids.pkl'
ukb_ids = pkl.load(open(pkl_file, 'rb'))

len(ukb_ids)

In [ ]:
eur_ids = pl.DataFrame({
    "FID": ukb_ids,
    "IID": ukb_ids
})

eur_ids.write_csv(
    '/s/project/geno2pheno/BFuncRVP/data/regenie/regenie_input/3rd_degree_unrelated_EUR_sample_500k.txt',
    separator=' ',
    include_header=False
)

In [ ]:
eur_ids

## Phenotypes

In [ ]:
pdf = pl.read_parquet('/s/project/deeprvat/ukb_gym/phenotypes/phenotypes190_missing80_unique2.parquet').rename({'individual': 'IID'})
pdf

In [ ]:
apdf = pl.read_csv('/s/project/geno2pheno/BFuncRVP/data/regenie/regenie_input/genebass_continuous_phenos.txt', separator=' ', ignore_errors=True, null_values=['NA'])
apdf = apdf.rename({col: col.lower() for col in apdf.columns}).rename({'fid':'FID', 'iid':'IID'})

apdf

In [ ]:
diff_cols = list(set(pdf.columns) - set(apdf.columns))
len(diff_cols)

In [ ]:
bin_phenos = pdf.with_columns(
    pl.col('IID').alias('FID')
).select(['FID', 'IID'] + diff_cols)

bin_phenos

In [ ]:
# bin_phenos = bin_phenos.fill_null(pl.lit("NA"))
# bin_phenos.write_csv(
#     '/s/project/geno2pheno/BFuncRVP/data/regenie/regenie_input/jurgens_binary_phenos.txt',
#     separator=' '
# )

In [ ]:
# pheno = apdf['mean_corpuscular_haemoglobin']
pheno = apdf['mothers_age_at_death']

plt.hist(pheno, bins=100)
plt.show()

In [ ]:
from scipy.stats import norm

def inverse_normal_transform(df: pl.LazyFrame, cols: list[str]) -> pl.LazyFrame:
    out = df
    for col in tqdm(cols):
        # compute n as number of non-nulls in the column
        n = df.select(pl.col(col).drop_nulls().count()).collect().item()

        out = out.with_columns(
            (
                (pl.col(col).rank(method="average", descending=False, seed=0) - 0.5) / n
            ).alias(f"{col}_p")
        )

        out = out.with_columns(
            pl.col(f"{col}_p")
            .map_batches(lambda s: pl.Series(norm.ppf(s.to_numpy())), return_dtype=pl.Float64)
            .alias(f"{col}_int")
        )
    return out


In [ ]:
cols = set(apdf.columns) - {'FID', 'IID'}
df_int = inverse_normal_transform(apdf.lazy(), cols).collect()
df_int = df_int.select(
    ['FID', 'IID'] + [c for c in df_int.columns if c.endswith("_int")]
)

df_int

In [ ]:
# df_int.write_csv(
#     '/s/project/geno2pheno/BFuncRVP/data/regenie/regenie_input/genebass_continuous_phenos_INT.txt',
#     separator=' '
# )

In [ ]:
allphenos = df_int.join(bin_phenos, on=['FID', 'IID'])
allphenos = allphenos.with_columns([
    pl.col(col).cast(pl.Float32) if dtype == pl.Float64
    else pl.col(col).cast(pl.Int32) if dtype == pl.Int64
    else pl.col(col)
    for col, dtype in allphenos.schema.items()
])

# allphenos.write_parquet('/s/project/deeprvat/ukb_gym/phenotypes/phenotypes190_missing20_unique2_int.parquet')
allphenos

## Fix REGENIE output paths

In [ ]:
fl = pl.read_csv(
    # '/s/project/deeprvat/ukb_gym/regenie/quant_traits/prs.list_pred.list',
    '/s/project/deeprvat/ukb_gym/regenie/binary_traits/prs.list_prs.list',
    separator=' ',
    has_header=False
).rename({"column_1": "col1", "column_2": "col2"})

fl = fl.with_columns(
    pl.col("col2").map_elements(
        # lambda path: "/".join(path.split("/")[:-2] + ["quant_traits"] + path.split("/")[-1:]),
        lambda path: "/".join(["/s/project/deeprvat/ukb_gym/regenie/binary_traits"] + path.split("/")[-1:]),
        return_dtype=pl.Utf8
    ).alias("col2_mod")
)

fl

In [ ]:
fl['col2_mod'][0]

In [ ]:
fl[['col1', 'col2_mod']].write_csv(
    # '/s/project/deeprvat/ukb_gym/regenie/quant_traits/prs.list_pred.list',
    '/s/project/deeprvat/ukb_gym/regenie/binary_traits/prs.list_prs.list',
    separator=' ',
    include_header=False
)

## Burden scores

In [ ]:
import pyranges as pr

gtf_file = '/s/project/deeprvat/deeprvat_input/gencode.v38.basic.annotation.gtf.gz'

gene_pos = pr.read_gtf(gtf_file)
gene_pos = gene_pos[
    (gene_pos.Feature == "gene") & (gene_pos.gene_type == "protein_coding")
][["Chromosome", "Start", "End", "gene_id"]].as_df()

gene_meta = pl.from_pandas(gene_pos)
gene_meta

In [ ]:
gene_meta.with_columns(
    pl.col('gene_id').str.split('.').list.first().alias('gene_name')
)

In [ ]:
from bgen import BgenReader, BgenWriter

# bgen_file = '/s/project/geno2pheno/BFuncRVP/data/regenie/regenie_input/scores_bgen.bgen'
bgen_file = '/s/project/deeprvat/ukb_gym/phenotypes/regenie/scores_bgen.bgen'

bfile = BgenReader(bgen_file)
rsids = bfile.rsids()

# select a variant by indexing
var = bfile[100]

# pull out genotype probabilities
probs = var.probabilities  # returns 2D numpy array
dosage = var.minor_allele_dosage  # returns 1D numpy array for biallelic variant

print(probs, dosage)

In [ ]:
probs.shape

In [ ]:
bfile.rsids()

In [ ]:
g = pl.read_csv('/s/project/deeprvat/ukb_gym/experimental_assays/dms_genes.txt', has_header=False)
g

In [ ]:
set(g['column_1'].to_list()) - set(bfile.rsids())